# 70. The Minimum Spanning Tree Problem

## Tier 2: The Classic Heuristic (Python Implementation)

### Key assumptions
- We can use greedy algorithms to find the optimal MST efficiently
- Kruskal's algorithm with Union-Find data structure provides optimal solution
- Time complexity is dominated by sorting edges: O(E log E)
- Union-Find with path compression enables efficient cycle detection

### Approach (step-by-step)
1. **Sort all edges** by weight in ascending order
2. **Initialize Union-Find** structure for cycle detection
3. **Iterate through sorted edges**: add each edge if it doesn't create a cycle
4. **Track selected edges** until we have n-1 edges
5. **Return the MST** with total cost

### What to look for in the results
- Step-by-step edge selection process
- Union-Find operations (find and union)
- Cycle detection in action
- Final MST structure and total cost
- Comparison with mathematical formulation (Tier 1)

### Concrete example (from the source)
We'll implement Kruskal's algorithm on the 8-city network:
- Cities: A, B, C, D, E, F, G, H
- Expected MST cost: $36 million
- Expected edges: 7 edges (n-1 for 8 cities)

### Why this Tier exists vs Tier 1
Tier 2 provides an efficient, practical implementation that runs in polynomial time, unlike the exponential complexity of the mathematical formulation approach. Kruskal's algorithm is the standard method used in practice for MST problems.

### Pros / Cons vs Tier 1
**Pros vs Tier 1:**
- Much faster: O(E log E) vs exponential time
- No need for optimization solvers
- Simple to implement and understand
- Guaranteed optimal solution for MST problems
- Works efficiently on large networks

**Cons vs Tier 1:**
- Less mathematical rigor
- Doesn't provide sensitivity analysis capabilities
- Harder to modify for constrained variants

### When to use this Tier
- Large-scale networks where performance matters
- Practical implementations in production systems
- When you need a fast, reliable MST solution
- Standard MST problems without additional constraints

In [ ]:
# Import required packages for Kruskal's algorithm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Edge:
    """Represents an edge in the graph"""
    u: int  # from vertex index
    v: int  # to vertex index
    weight: float
    u_name: str  # from vertex name
    v_name: str  # to vertex name

@dataclass
class KruskalStep:
    """Represents a step in Kruskal's algorithm"""
    step: int
    edge: Edge
    action: str  # "Added" or "Skipped"
    reason: str  # Why the action was taken
    mst_edges: List[Edge]
    total_cost: float
    parent: List[int]
    rank: List[int]

In [ ]:
class UnionFind:
    """Union-Find data structure with path compression and union by rank"""
    
    def __init__(self, n: int):
        # Initially, each vertex is its own parent (disjoint set)
        self.parent = list(range(n))
        # Rank helps keep trees shallow
        self.rank = [0] * n
        self.operations = []  # Track operations for visualization
    
    def find(self, x: int) -> int:
        """Find the root of vertex x with path compression"""
        if self.parent[x] != x:
            # Path compression: make x point directly to root
            old_parent = self.parent[x]
            self.parent[x] = self.find(self.parent[x])
            self.operations.append(f"find({x}): {x} -> {old_parent} -> {self.parent[x]}")
        else:
            self.operations.append(f"find({x}): {x} is root")
        return self.parent[x]
    
    def union(self, x: int, y: int) -> bool:
        """Union the sets containing x and y. Returns True if merged, False if already in same set"""
        px = self.find(x)
        py = self.find(y)
        
        if px == py:
            # Already in the same set (would create a cycle)
            self.operations.append(f"union({x},{y}): {x} and {y} already connected (cycle detected)")
            return False
        
        # Union by rank: attach smaller tree to larger tree
        if self.rank[px] < self.rank[py]:
            self.parent[px] = py
            self.operations.append(f"union({x},{y}): attach {px} to {py} (rank {self.rank[px]} < {self.rank[py]})")
        elif self.rank[px] > self.rank[py]:
            self.parent[py] = px
            self.operations.append(f"union({x},{y}): attach {py} to {px} (rank {self.rank[px]} > {self.rank[py]})")
        else:
            # Same rank, choose one as new root and increment its rank
            self.parent[py] = px
            self.rank[px] += 1
            self.operations.append(f"union({x},{y}): attach {py} to {px} (equal rank, increment to {self.rank[px]})")
        
        return True

In [ ]:
def create_8_city_network() -> Tuple[List[str], List[Edge]]:
    """Create the 8-city network from the problem statement"""
    cities = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    
    # Define edges based on the cost matrix
    edges = [
        Edge(0, 1, 12, 'A', 'B'),
        Edge(0, 2, 8, 'A', 'C'),
        Edge(0, 3, 6, 'A', 'D'),
        Edge(1, 2, 5, 'B', 'C'),
        Edge(1, 7, 7, 'B', 'H'),
        Edge(2, 3, 9, 'C', 'D'),
        Edge(3, 4, 4, 'D', 'E'),
        Edge(4, 6, 3, 'E', 'G'),
        Edge(5, 6, 5, 'F', 'G'),
        Edge(5, 7, 6, 'F', 'H'),
        Edge(6, 7, 8, 'G', 'H')
    ]
    
    return cities, edges

In [ ]:
def kruskal_mst_with_steps(cities: List[str], edges: List[Edge]) -> Tuple[List[Edge], float, List[KruskalStep]]:
    """Kruskal's algorithm with detailed step tracking"""
    n = len(cities)
    uf = UnionFind(n)
    
    # Sort edges by weight (ascending)
    sorted_edges = sorted(edges, key=lambda e: e.weight)
    
    mst_edges = []
    total_cost = 0
    steps = []
    
    print("Kruskal's Algorithm - Step by Step:")
    print("="*50)
    print(f"Sorted edges by weight: {[(e.u_name, e.v_name, e.weight) for e in sorted_edges]}")
    print()
    
    for step_idx, edge in enumerate(sorted_edges):
        if len(mst_edges) >= n - 1:
            break  # MST complete
        
        # Clear operations log for this step
        uf.operations = []
        
        # Try to add this edge
        if uf.union(edge.u, edge.v):
            # Edge added to MST
            mst_edges.append(edge)
            total_cost += edge.weight
            action = "Added"
            reason = "No cycle created"
        else:
            # Edge skipped (would create cycle)
            action = "Skipped"
            reason = "Would create cycle"
        
        # Record step
        step = KruskalStep(
            step=step_idx + 1,
            edge=edge,
            action=action,
            reason=reason,
            mst_edges=mst_edges.copy(),
            total_cost=total_cost,
            parent=uf.parent.copy(),
            rank=uf.rank.copy()
        )
        steps.append(step)
        
        # Print step details
        print(f"Step {step_idx + 1}: Edge {edge.u_name}-{edge.v_name} (weight={edge.weight})")
        print(f"  Action: {action}")
        print(f"  Reason: {reason}")
        print(f"  MST edges: {[(e.u_name, e.v_name, e.weight) for e in mst_edges]}")
        print(f"  Total cost: {total_cost}")
        
        # Show Union-Find operations
        if uf.operations:
            print(f"  Union-Find operations:")
            for op in uf.operations:
                print(f"    {op}")
        
        print(f"  Parent array: {uf.parent}")
        print(f"  Rank array: {uf.rank}")
        print()
    
    return mst_edges, total_cost, steps

In [ ]:
def visualize_kruskal_steps(cities: List[str], edges: List[Edge], steps: List[KruskalStep]):
    """Visualize the step-by-step execution of Kruskal's algorithm"""
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    # Create positions for cities (circular layout)
    n = len(cities)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    positions = {city: (np.cos(angle), np.sin(angle)) for city, angle in zip(cities, angles)}
    
    # Plot each step
    for i, ax in enumerate(axes):
        if i >= len(steps):
            ax.set_visible(False)
            continue
        
        step = steps[i]
        
        # Plot all potential edges in light gray
        for edge in edges:
            pos1 = positions[edge.u_name]
            pos2 = positions[edge.v_name]
            ax.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], 'lightgray', alpha=0.5, linewidth=1)
            ax.text((pos1[0]+pos2[0])/2, (pos1[1]+pos2[1])/2, f'{edge.weight}', 
                    fontsize=8, ha='center', alpha=0.5)
        
        # Plot MST edges in blue
        for edge in step.mst_edges:
            pos1 = positions[edge.u_name]
            pos2 = positions[edge.v_name]
            ax.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], 'blue', linewidth=3)
        
        # Highlight current edge
        edge = step.edge
        pos1 = positions[edge.u_name]
        pos2 = positions[edge.v_name]
        color = 'green' if step.action == 'Added' else 'red'
        ax.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], color, linewidth=4, alpha=0.7)
        
        # Plot cities
        for city, pos in positions.items():
            ax.plot(pos[0], pos[1], 'o', markersize=8, markerfacecolor='lightblue', 
                    markeredgecolor='navy', markeredgewidth=2)
            ax.text(pos[0], pos[1]-0.15, city, fontsize=10, ha='center', fontweight='bold')
        
        ax.set_title(f'Step {step.step}: {step.edge.u_name}-{step.edge.v_name} ({step.action})\n'
                    f'Cost: ${step.total_cost}M', fontsize=12, fontweight='bold')
        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-1.5, 1.5)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def analyze_kruskal_performance(cities: List[str], edges: List[Edge], mst_edges: List[Edge], total_cost: float):
    """Analyze the performance and characteristics of Kruskal's algorithm"""
    print("="*60)
    print("KRUSKAL'S ALGORITHM - PERFORMANCE ANALYSIS")
    print("="*60)
    
    # Basic results
    print(f"\n🎯 FINAL RESULTS:")
    print(f"   MST edges selected: {len(mst_edges)}")
    print(f"   Total cost: ${total_cost} million")
    print(f"   Algorithm completed successfully: {'✓' if len(mst_edges) == len(cities) - 1 else '✗'}")
    
    # Edge selection analysis
    print(f"\n📊 EDGE SELECTION ANALYSIS:")
    print(f"   Total edges considered: {len(edges)}")
    print(f"   Edges selected: {len(mst_edges)}")
    print(f"   Edges rejected: {len(edges) - len(mst_edges)}")
    print(f"   Selection ratio: {len(mst_edges)/len(edges)*100:.1f}%")
    
    # Cost efficiency
    total_possible_cost = sum(edge.weight for edge in edges)
    efficiency = total_cost / total_possible_cost * 100
    print(f"\n💰 COST EFFICIENCY:")
    print(f"   Total possible cost: ${total_possible_cost} million")
    print(f"   MST cost: ${total_cost} million")
    print(f"   Cost savings: ${total_possible_cost - total_cost} million")
    print(f"   Efficiency ratio: {efficiency:.1f}%")
    
    # Algorithm complexity analysis
    print(f"\n⚡ ALGORITHM COMPLEXITY:")
    print(f"   Time complexity: O(E log E) where E = {len(edges)}")
    print(f"   Space complexity: O(V) where V = {len(cities)}")
    print(f"   Dominant operation: Sorting edges ({len(edges)} log {len(edges)})")
    
    # Union-Find operations analysis
    print(f"\n🔗 UNION-FIND OPERATIONS:")
    print(f"   Each union operation: α(V) ≈ constant (inverse Ackermann)")
    print(f"   Total find operations: ~2 × {len(edges)} = {2*len(edges)}")
    print(f"   Total union operations: ~{len(mst_edges)}")
    print(f"   Path compression enabled: ✓")
    print(f"   Union by rank enabled: ✓")
    
    # Comparison with expected solution
    print(f"\n🎯 SOLUTION VERIFICATION:")
    expected_cost = 36
    expected_edges = 7
    
    print(f"   Expected total cost: ${expected_cost} million")
    print(f"   Actual total cost: ${total_cost} million")
    print(f"   Cost match: {'✓' if total_cost == expected_cost else '✗'}")
    
    print(f"   Expected number of edges: {expected_edges}")
    print(f"   Actual number of edges: {len(mst_edges)}")
    print(f"   Edge count match: {'✓' if len(mst_edges) == expected_edges else '✗'}")
    
    if total_cost == expected_cost and len(mst_edges) == expected_edges:
        print(f"\n🎉 PERFECT! Kruskal's algorithm found the optimal solution!")
        print(f"   This demonstrates that greedy algorithms can be optimal for MST problems.")
    else:
        print(f"\n⚠️  Solution differs from expected result")
    
    # Edge weight distribution in MST
    print(f"\n📈 MST EDGE WEIGHT DISTRIBUTION:")
    weights = [edge.weight for edge in mst_edges]
    print(f"   Minimum edge weight: ${min(weights)} million")
    print(f"   Maximum edge weight: ${max(weights)} million")
    print(f"   Average edge weight: ${np.mean(weights):.1f} million")
    print(f"   Standard deviation: ${np.std(weights):.1f} million")
    
    # Node degree analysis
    degrees = {city: 0 for city in cities}
    for edge in mst_edges:
        degrees[edge.u_name] += 1
        degrees[edge.v_name] += 1
    
    print(f"\n🌐 NODE DEGREE ANALYSIS:")
    for city, degree in sorted(degrees.items()):
        print(f"   {city}: {degree} connections")
    
    max_degree = max(degrees.values())
    min_degree = min(degrees.values())
    print(f"   Degree range: {min_degree} to {max_degree}")
    
    # Performance comparison with theoretical bounds
    print(f"\n🏆 PERFORMANCE COMPARISON:")
    print(f"   Theoretical minimum edges: {len(cities) - 1}")
    print(f"   Actual edges: {len(mst_edges)}")
    print(f"   Optimality: {'Guaranteed for MST' if len(mst_edges) == len(cities) - 1 else 'Not optimal'}")
    print(f"   Algorithm type: Greedy (optimal for MST)")
    print(f"   Practical performance: Excellent for large networks")

In [ ]:
def compare_with_mathematical_formulation():
    """Compare Kruskal's algorithm with mathematical formulation approach"""
    print("\n" + "="*60)
    print("COMPARISON: KRUSKAL vs MATHEMATICAL FORMULATION")
    print("="*60)
    
    comparison_data = [
        ("Time Complexity", "O(E log E)", "Exponential"),
        ("Space Complexity", "O(V)", "O(V² + 2^V)"),
        ("Solution Quality", "Optimal", "Optimal"),
        ("Implementation", "Simple", "Complex"),
        ("Scalability", "Excellent", "Poor"),
        ("Dependencies", "None", "MILP Solver"),
        ("Flexibility", "Limited", "High"),
        ("Speed", "Fast", "Slow"),
        ("Memory Usage", "Low", "High"),
        ("Practical Use", "High", "Low")
    ]
    
    print(f"{'Aspect':<20} {'Kruskal':<15} {'Mathematical':<15}")
    print("-" * 50)
    for aspect, kruskal, math_form in comparison_data:
        print(f"{aspect:<20} {kruskal:<15} {math_form:<15}")
    
    print("\n📋 KEY INSIGHTS:")
    print("   • Kruskal's algorithm is vastly superior for practical MST problems")
    print("   • Mathematical formulation is useful for constrained variants")
    print("   • Both guarantee optimal solutions for standard MST problems")
    print("   • Kruskal's algorithm is the preferred choice for production systems")
    
    print("\n🎯 RECOMMENDATIONS:")
    print("   • Use Kruskal's algorithm for standard MST problems")
    print("   • Use mathematical formulation for constrained MST variants")
    print("   • Use Kruskal's algorithm as a benchmark for testing new methods")
    print("   • Consider mathematical formulation when you need sensitivity analysis")

In [ ]:
# Main execution - run Kruskal's algorithm
print("🚀 EXECUTING KRUSKAL'S ALGORITHM FOR MINIMUM SPANNING TREE")
print("="*60)

# Create the network
cities, edges = create_8_city_network()

print(f"\n📊 PROBLEM SETUP:")
print(f"   Cities: {cities}")
print(f"   Number of vertices: {len(cities)}")
print(f"   Number of edges: {len(edges)}")
print(f"   Expected MST edges: {len(cities) - 1}")
print(f"   Expected optimal cost: $36 million")

# Run Kruskal's algorithm with step tracking
mst_edges, total_cost, steps = kruskal_mst_with_steps(cities, edges)

# Analyze performance
analyze_kruskal_performance(cities, edges, mst_edges, total_cost)

# Compare with mathematical formulation
compare_with_mathematical_formulation()

# Visualize the algorithm steps
print(f"\n🎨 VISUALIZATION: KRUSKAL'S ALGORITHM STEP-BY-STEP")
visualize_kruskal_steps(cities, edges, steps)

In [ ]:
# Additional analysis: Edge rejection analysis
def edge_rejection_analysis():
    """Analyze which edges were rejected and why"""
    print("\n" + "="*60)
    print("EDGE REJECTION ANALYSIS")
    print("="*60)
    
    # Get all edges that were skipped
    skipped_edges = [step.edge for step in steps if step.action == 'Skipped']
    
    print(f"\n🚫 EDGES REJECTED (would create cycles):")
    for edge in skipped_edges:
        print(f"   {edge.u_name}-{edge.v_name} (weight: ${edge.weight}M)")
    
    print(f"\n📊 REJECTION STATISTICS:")
    print(f"   Total edges considered: {len(edges)}")
    print(f"   Edges accepted: {len(mst_edges)}")
    print(f"   Edges rejected: {len(skipped_edges)}")
    print(f"   Rejection rate: {len(skipped_edges)/len(edges)*100:.1f}%")
    
    # Analyze rejected edge weights
    rejected_weights = [edge.weight for edge in skipped_edges]
    accepted_weights = [edge.weight for edge in mst_edges]
    
    print(f"\n💰 WEIGHT ANALYSIS:")
    print(f"   Average rejected edge weight: ${np.mean(rejected_weights):.1f}M")
    print(f"   Average accepted edge weight: ${np.mean(accepted_weights):.1f}M")
    print(f"   Weight difference: ${np.mean(rejected_weights) - np.mean(accepted_weights):.1f}M")
    
    # When were edges rejected?
    print(f"\n⏰ TIMING ANALYSIS:")
    for i, step in enumerate(steps):
        if step.action == 'Skipped':
            print(f"   Step {step.step}: {step.edge.u_name}-{step.edge.v_name} rejected at cost ${step.total_cost}M")

# Run edge rejection analysis
edge_rejection_analysis()